# Carnet d'analyses : dégradation de la végétation autour des sentiers

**Projet PI² n°192 · équipe P192_T4 · ESILV / De Vinci Research Center**

Équipe : Axel Bonneau, Sina Balaei Langroodi, Jonas Barth, Florian Berger, Kuresa Ramen, Enzo Sierra-Guisset

> **Note interne, à supprimer avant diffusion.** Ce squelette fournit le code, les cartes et les tableaux.
> Les textes d'objet, d'interprétation, de limites et de conclusion sont à rédiger par l'équipe,
> avec vos propres mots (règle des 30 % de contenu généré par IA).

## Objet du carnet

_À rédiger : la question étudiée, pourquoi elle compte, ce que montre ce carnet (5 à 8 lignes)._

## Lire et relancer ce carnet

- **Lire** : sur GitHub, les cartes et les graphiques sont déjà affichés sous chaque cellule.
- **Relancer** : ouvrir le carnet dans Google Colab (Fichier › Ouvrir un notebook › GitHub), puis Exécution › Tout exécuter.
  Il faut un compte Google ayant accès au projet Earth Engine `degradation-vegetables`.
- **Modifier les zones, années ou bandes** : uniquement dans la cellule « Paramètres » ci-dessous et dans `config/config.yaml`.
- Les figures sont enregistrées dans `figures/` et les tableaux dans `resultats/`, pour être réutilisés dans le diaporama et le rapport.

## 0. Préparation

In [ ]:
# Récupération du code (Colab) ou placement à la racine du dépôt (Jupyter local)
import os, subprocess, sys

DEPOT = "https://github.com/axel12334556/vegetation-sentiers.git"   # adresse du dépôt (public)

if "google.colab" in sys.modules and not os.path.exists("src/analyses.py"):
    if not os.path.exists("vegetation-sentiers"):
        subprocess.run(["git", "clone", "--quiet", DEPOT], check=True)
    os.chdir("vegetation-sentiers")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Dossier de travail :", os.getcwd())

In [ ]:
# Paramètres du carnet : c'est ici que l'on choisit zones, années et sentiers
PROJET = "degradation-vegetables"
ZONES = ["calanques", "ventoux", "fontainebleau"]       # noms définis dans config/config.yaml

# Assets Earth Engine des sentiers OpenStreetMap (None tant qu'ils ne sont pas importés)
SENTIERS = {
    "calanques": "projects/degradation-vegetables/assets/sentiers_calanques",
    "ventoux": "projects/degradation-vegetables/assets/sentiers_ventoux",
    "fontainebleau": "projects/degradation-vegetables/assets/sentiers_fontainebleau",
}

# Zones perturbées à exclure (incendie, chantier, repousse) : polygone My Maps en .kml dans zones/
EXCLUSIONS = {
    "calanques": None,        # ex. "zones/calanques_repousse.kml"
    "ventoux": None,
    "fontainebleau": None,
}

ANNEE_REF = 2024                                  # été de référence pour l'état de la végétation
PERIODES = [(2019, 2024), (2017, 2024), (2019, 2022)]   # (avant, après) ; la première sert de période principale
ANNEES = list(range(2017, 2026))                  # années inventoriées dans la section 1
PROCHE, TEMOIN = 15, (50, 100)                    # bandes sentier et témoin, en mètres
SEUIL_BAISSE = -0.1                               # "forte baisse" : évolution du NDVI inférieure à ce seuil
N_TRONCONS = 400                                  # tronçons tirés au hasard pour le test de robustesse

In [ ]:
# Connexion à Earth Engine et chargement des outils
import ee
try:
    ee.Initialize(project=PROJET)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJET)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.config import load_config, load_zones
from src.kml import read_polygon
from src import analyses as A

pd.options.display.float_format = "{:.4f}".format
os.makedirs("figures", exist_ok=True)
os.makedirs("resultats", exist_ok=True)

def sauver(fig, nom):
    """Enregistre une figure dans figures/ (pour le diaporama et le rapport)."""
    fig.savefig(f"figures/{nom}.png", dpi=150, bbox_inches="tight")

def grille(n, largeur=6, hauteur=5):
    fig, axes = plt.subplots(1, n, figsize=(largeur * n, hauteur))
    return fig, np.atleast_1d(axes)

In [ ]:
# Zones d'étude, sentiers et exclusions
cfg = load_config("config/config.yaml")
zones = {z.name: z for z in load_zones(cfg) if z.name in ZONES}
regions = {nom: A.polygone(z.ring) for nom, z in zones.items()}
exclusions = {nom: (A.polygone(read_polygon(EXCLUSIONS[nom])) if EXCLUSIONS.get(nom) else None) for nom in regions}
sentiers = {nom: ee.FeatureCollection(SENTIERS[nom]) for nom in regions if SENTIERS.get(nom)}

resume_zones = pd.DataFrame({
    "surface_km2": {n: round(z.area_km2(), 1) for n, z in zones.items()},
    "troncons_sentiers": {n: (sentiers[n].size().getInfo() if n in sentiers else 0) for n in zones},
    "zone_exclue": {n: exclusions[n] is not None for n in zones},
})
resume_zones

## 1. Données

| Élément | Choix |
| --- | --- |
| Images | Sentinel-2 niveau 2A harmonisé (`COPERNICUS/S2_SR_HARMONIZED`), pixel de 10 m |
| Période | été, du 15 juin au 31 août, chaque année |
| Nuages | images à moins de 40 % de nuages, puis masque pixel par pixel : seules les classes SCL 4 (végétation) et 5 (sol nu) sont gardées |
| Composite | médiane des images de l'été |
| Sentiers | OpenStreetMap, `highway` = path, footway, steps, bridleway, extraits avec Overpass Turbo le [date à compléter] |

Indices calculés (B2 bleu, B3 vert, B4 rouge, B8 proche infrarouge, B11 infrarouge moyen) :

$$NDVI = \frac{B8 - B4}{B8 + B4} \qquad GNDVI = \frac{B8 - B3}{B8 + B3} \qquad BSI = \frac{(B11 + B4) - (B8 + B2)}{(B11 + B4) + (B8 + B2)}$$

In [ ]:
# Nombre d'images Sentinel-2 utilisables par zone et par été
images = A.nombre_images(regions, ANNEES)
images.to_csv("resultats/nombre_images.csv")
images

### Interprétation : disponibilité des images

> **À rédiger par l'équipe.** Auteur·e : [Prénom]

Questions pour guider la rédaction (à supprimer ensuite) :
- Combien d'images par été et par zone ? Une année ou une zone est-elle nettement moins couverte ?
- Ce nombre suffit-il pour une médiane fiable ?

## 2. Analyse 1 : état de la végétation en été

**Question** : quel est le niveau de végétation (NDVI) dans chaque zone pendant l'été de référence ?

In [ ]:
etat = {nom: A.composite_saison(reg, ANNEE_REF) for nom, reg in regions.items()}

fig, axes = grille(len(regions))
for ax, (nom, reg) in zip(axes, regions.items()):
    A.carte(ax, A.masquer_zone(etat[nom].select("NDVI"), exclusions[nom]), A.VIS_NDVI, reg,
            f"{nom} : NDVI été {ANNEE_REF}", legende="NDVI")
sauver(fig, f"a1_ndvi_{ANNEE_REF}")
plt.show()

stats_etat = pd.DataFrame({
    nom: A.statistiques(A.masquer_zone(etat[nom], exclusions[nom]), "NDVI", reg)
    for nom, reg in regions.items()
}).T
stats_etat.to_csv(f"resultats/a1_stats_ndvi_{ANNEE_REF}.csv")
stats_etat

### Interprétation : état de la végétation

> **À rédiger par l'équipe.** Auteur·e : [Prénom]

Questions pour guider la rédaction (à supprimer ensuite) :
- Quelle zone est la plus végétalisée ? Les écarts sont-ils cohérents avec le type de milieu (forêt, garrigue, montagne) ?
- Voit-on à l'œil des lignes plus claires qui pourraient être des sentiers ?

## 3. Analyse 2 : évolution de la végétation entre deux étés

**Question** : où la végétation a-t-elle baissé ou progressé entre les deux étés de la période principale ?
Une évolution négative (marron) signale une baisse du NDVI, une évolution positive (bleu-vert) une hausse.

In [ ]:
avant, apres = PERIODES[0]
evol = {nom: A.masquer_zone(A.evolution(reg, avant, apres), exclusions[nom]) for nom, reg in regions.items()}

fig, axes = grille(len(regions))
for ax, (nom, reg) in zip(axes, regions.items()):
    A.carte(ax, evol[nom], A.VIS_EVOLUTION, reg, f"{nom} : évolution NDVI {avant} → {apres}",
            legende="évolution du NDVI")
sauver(fig, f"a2_evolution_{avant}_{apres}")
plt.show()

stats_evol = pd.DataFrame({
    nom: {**A.statistiques(evol[nom], "dNDVI", reg), **A.parts_changement(evol[nom], reg)}
    for nom, reg in regions.items()
}).T
stats_evol.to_csv(f"resultats/a2_stats_evolution_{avant}_{apres}.csv")
stats_evol

In [ ]:
# Répartition des évolutions : une distribution centrée sur 0 indique une zone globalement stable
fig, ax = plt.subplots(figsize=(8, 4))
for nom, reg in regions.items():
    h = A.histogramme(evol[nom], "dNDVI", reg, -0.5, 0.5)
    ax.plot(h["valeur"], h["pixels"] / h["pixels"].sum(), label=nom)
ax.axvline(0, color="#999999", linewidth=0.8)
ax.set_xlabel(f"Évolution du NDVI {avant} → {apres}")
ax.set_ylabel("Part des pixels")
ax.set_title("Répartition des évolutions par zone")
ax.legend()
ax.grid(alpha=0.3)
sauver(fig, f"a2_histogramme_{avant}_{apres}")
plt.show()

### Interprétation : évolution entre deux étés

> **À rédiger par l'équipe.** Auteur·e : [Prénom]

Questions pour guider la rédaction (à supprimer ensuite) :
- La végétation a-t-elle globalement progressé, baissé, ou est-elle stable ?
- Quelles grandes taches apparaissent (repousse après incendie, chantier, urbanisation) ? Faut-il les exclure via EXCLUSIONS ?
- Quelle part de chaque zone est très perturbée (colonne part_perturbee) ?

## 4. Analyse 3 : la végétation près des sentiers

**Question** : la végétation est-elle plus faible, ou a-t-elle davantage reculé, près des sentiers que plus loin ?

**Méthode** : pour chaque pixel, distance au sentier OpenStreetMap le plus proche (jusqu'à 150 m), regroupée par classes de 10 m.
On compare une **bande sentier** (0 à 15 m) et une **bande témoin** (50 à 100 m).
Deux angles : l'**état** (NDVI de l'été de référence), qui reflète la dégradation accumulée, et l'**évolution** entre deux étés, qui reflète une dégradation récente.

In [ ]:
if not sentiers:
    print("Aucun asset de sentiers renseigné dans SENTIERS : section ignorée.")
distances = {nom: A.distance_sentiers(sentiers[nom], regions[nom]) for nom in sentiers}

# Carte : évolution de la période principale avec les sentiers en rouge
fig, axes = grille(len(sentiers) or 1)
for ax, nom in zip(axes, sentiers):
    A.carte(ax, evol[nom], A.VIS_EVOLUTION, regions[nom], f"{nom} : évolution {avant} → {apres} et sentiers",
            legende="évolution du NDVI", sentiers=sentiers[nom])
sauver(fig, f"a3_sentiers_evolution_{avant}_{apres}")
plt.show()

In [ ]:
# Angle 1 : état de la végétation selon la distance au sentier
profils_etat, lignes_etat = {}, []
for nom in sentiers:
    img = A.masquer_zone(etat[nom], exclusions[nom])
    profils_etat[nom] = A.profil_distance(img, "NDVI", distances[nom], regions[nom])
    lignes_etat.append({"zone": nom, "indice": "NDVI", "periode": str(ANNEE_REF),
                        **A.sentier_vs_temoin(img, "NDVI", distances[nom], regions[nom], PROCHE, TEMOIN)})

ax = A.tracer_profils(profils_etat, f"NDVI de l'été {ANNEE_REF} selon la distance au sentier", "NDVI moyen")
sauver(ax.figure, f"a3_profil_etat_{ANNEE_REF}")
plt.show()
pd.DataFrame(lignes_etat)

In [ ]:
# Angle 2 : évolution selon la distance au sentier, pour chaque période
lignes_evol = []
for nom in sentiers:
    profils = {}
    for (a, b) in PERIODES:
        diff = A.masquer_zone(A.evolution(regions[nom], a, b), exclusions[nom])
        profils[f"{a} → {b}"] = A.profil_distance(diff, "dNDVI", distances[nom], regions[nom])
        lignes_evol.append({"zone": nom, "indice": "dNDVI", "periode": f"{a}-{b}",
                            **A.sentier_vs_temoin(diff, "dNDVI", distances[nom], regions[nom], PROCHE, TEMOIN)})
    ax = A.tracer_profils(profils, f"{nom} : évolution du NDVI selon la distance au sentier", "Évolution moyenne du NDVI")
    sauver(ax.figure, f"a3_profil_evolution_{nom}")
    plt.show()

resultats_sentiers = pd.DataFrame(lignes_etat + lignes_evol)
resultats_sentiers.to_csv("resultats/a3_sentier_vs_temoin.csv", index=False)
resultats_sentiers

### Interprétation : végétation près des sentiers

> **À rédiger par l'équipe.** Auteur·e : [Prénom]

Questions pour guider la rédaction (à supprimer ensuite) :
- État : le NDVI est-il plus bas dans la bande sentier que dans la bande témoin ? De combien ?
- Évolution : l'écart sentier − témoin est-il négatif, nul ou positif selon les périodes ? Est-il stable d'une période à l'autre ?
- Ces écarts sont-ils grands par rapport à la variabilité de la zone (écart-type de l'analyse 2) ?
- Quelles limites : sentiers absents d'OpenStreetMap, routes et pistes non prises en compte, densité du réseau ?

### Robustesse : le résultat dépasse-t-il le bruit ?

Les écarts sentier − témoin ci-dessus sont petits. Deux contrôles permettent de juger s'ils signifient quelque chose.

1. **Part de pixels en forte baisse** : au lieu d'une moyenne, on compte la part des pixels dont le NDVI a baissé de plus de `SEUIL_BAISSE` entre les deux étés, près des sentiers et dans la bande témoin. Une moyenne peut cacher quelques zones très dégradées.
2. **Écart par tronçon et intervalle de confiance** : `N_TRONCONS` tronçons de sentier (plus de 100 m) sont tirés au hasard. Pour chacun, on compare le NDVI à moins de 15 m du tronçon avec le NDVI de **son propre voisinage** (50 à 100 m, hors zone proche d'un autre sentier), ce qui compare des endroits de même altitude, de même exposition et de même sol. Un intervalle à 95 % est calculé par bootstrap : on rééchantillonne les **tronçons**, pas les pixels, car des pixels voisins se ressemblent. Si l'intervalle contient 0, l'écart ne se distingue pas du bruit.

In [ ]:
# Contrôle 1 : part de pixels en forte baisse, période principale
lignes_baisse = []
for nom in sentiers:
    lignes_baisse.append({"zone": nom, "periode": f"{avant}-{apres}", "seuil": SEUIL_BAISSE,
                          **A.part_baisse_sentier_temoin(evol[nom], distances[nom], regions[nom],
                                                         SEUIL_BAISSE, PROCHE, TEMOIN)})
parts_baisse = pd.DataFrame(lignes_baisse)
parts_baisse.to_csv("resultats/a3_part_forte_baisse.csv", index=False)
parts_baisse

In [ ]:
# Contrôle 2 : écart par tronçon et intervalle de confiance (peut prendre quelques minutes)
resumes = {"état": [], "évolution": []}
for nom in sentiers:
    for angle, image, bande in (("état", A.masquer_zone(etat[nom], exclusions[nom]), "NDVI"),
                                ("évolution", evol[nom], "dNDVI")):
        troncons = A.ecarts_par_troncon(image, bande, sentiers[nom], distances[nom], regions[nom],
                                        n=N_TRONCONS, proche=PROCHE, temoin=TEMOIN, exclusion=exclusions[nom])
        troncons.to_csv(f"resultats/a3_troncons_{angle}_{nom}.csv", index=False)
        resumes[angle].append({"zone": nom, **A.resumer_troncons(troncons)})

tableaux = {angle: pd.DataFrame(lignes) for angle, lignes in resumes.items()}
pd.concat({angle: t.set_index("zone") for angle, t in tableaux.items()}).to_csv("resultats/a3_robustesse.csv")

fig, axes = plt.subplots(1, 2, figsize=(13, 0.9 * len(sentiers) + 1.8))
A.tracer_intervalles(tableaux["état"], f"État (NDVI {ANNEE_REF})", "écart sentier − voisinage (NDVI)", axes[0])
A.tracer_intervalles(tableaux["évolution"], f"Évolution ({avant} → {apres})", "écart sentier − voisinage (dNDVI)", axes[1])
fig.suptitle("Écart moyen par tronçon et intervalle de confiance à 95 %")
fig.tight_layout()
sauver(fig, "a3_robustesse_troncons")
plt.show()
display(tableaux["état"])
display(tableaux["évolution"])

### Interprétation : robustesse du résultat

> **À rédiger par l'équipe.** Auteur·e : [Prénom]

Questions pour guider la rédaction (à supprimer ensuite) :
- La part de pixels en forte baisse est-elle plus élevée près des sentiers que dans le témoin ? Le rapport est-il proche de 1 ?
- Les intervalles de confiance contiennent-ils 0 ? Qu'en conclure : effet détecté, ou écart indiscernable du bruit ?
- La part de tronçons sous leur témoin est-elle proche de 50 % (hasard) ou nettement différente ?
- Les résultats sont-ils cohérents entre les trois zones ? Entre l'état et l'évolution ?
- Que ne peut-on pas conclure : dégradation moyenne sur toute la zone, effet local sur des sentiers très fréquentés ?

## 5. Analyse 4 : les sentiers sont-ils visibles sur l'image ? (groupe 2)

**Question** : à 10 m de résolution, l'indice de sol nu (BSI) fait-il ressortir les sentiers ? Sinon, faut-il une image plus fine (BD ORTHO de l'IGN, 20 cm) pour les détecter et entraîner un modèle ?
Un BSI plus élevé près des sentiers indiquerait qu'ils sont visibles dans les pixels Sentinel-2.

In [ ]:
nom_bsi = next(iter(sentiers), None)   # première zone disposant de sentiers
if nom_bsi:
    img_bsi = A.masquer_zone(etat[nom_bsi], exclusions[nom_bsi])
    fig, axes = grille(2, largeur=7)
    A.carte(axes[0], img_bsi.select("BSI"), A.VIS_BSI, regions[nom_bsi], f"{nom_bsi} : BSI été {ANNEE_REF}", legende="BSI")
    A.carte(axes[1], img_bsi.select("BSI"), A.VIS_BSI, regions[nom_bsi], "même carte avec les sentiers OpenStreetMap",
            legende="BSI", sentiers=sentiers[nom_bsi])
    sauver(fig, f"a4_bsi_{nom_bsi}_{ANNEE_REF}")
    plt.show()

    profil_bsi = A.profil_distance(img_bsi, "BSI", distances[nom_bsi], regions[nom_bsi])
    ax = A.tracer_profils({nom_bsi: profil_bsi}, f"BSI de l'été {ANNEE_REF} selon la distance au sentier", "BSI moyen")
    sauver(ax.figure, f"a4_profil_bsi_{nom_bsi}")
    plt.show()
    profil_bsi.to_csv(f"resultats/a4_profil_bsi_{nom_bsi}.csv", index=False)
    display(pd.DataFrame([A.sentier_vs_temoin(img_bsi, "BSI", distances[nom_bsi], regions[nom_bsi], PROCHE, TEMOIN)]))

### Dix plans rapprochés

Dix sites sont tirés au hasard sur les sentiers OpenStreetMap (sentiers de plus de 300 m, espacés d'au moins 800 m, hors zones exclues). Le tirage est fixe : les mêmes sites reviennent à chaque exécution, ce qui permet de comparer d'une version du carnet à l'autre.

Chaque site est montré dans une fenêtre de 600 m de côté, d'abord **sans** le sentier, puis **avec** : c'est la version sans sentier qui permet de juger s'il se voit vraiment à 10 m de résolution. Les coordonnées sont enregistrées dans `resultats/` pour retrouver les mêmes endroits dans QGIS (BD ORTHO, Esri) ou Google Earth.

In [ ]:
if nom_bsi:
    sites = A.choisir_plans(sentiers[nom_bsi], regions[nom_bsi], n=10, exclusion=exclusions[nom_bsi])
    sites.to_csv(f"resultats/a4_plans_{nom_bsi}.csv", index=False)
    display(sites)

    # Couleurs naturelles de l'été de référence
    rgb = A.composite_rgb(regions[nom_bsi], ANNEE_REF)
    fig = A.planches_sentiers(sites, rgb, A.VIS_RGB, sentiers[nom_bsi],
                              titre=f"{nom_bsi} : couleurs naturelles, été {ANNEE_REF}")
    sauver(fig, f"a4_plans_rgb_{nom_bsi}")
    plt.show()

In [ ]:
if nom_bsi:
    # Mêmes sites en indice de sol nu : le sentier ressort-il en rouge ?
    fig = A.planches_sentiers(sites, img_bsi.select("BSI"), A.VIS_BSI, sentiers[nom_bsi],
                              titre=f"{nom_bsi} : BSI, été {ANNEE_REF}")
    sauver(fig, f"a4_plans_bsi_{nom_bsi}")
    plt.show()

### Grille d'observation

À remplir à la main en regardant les planches ci-dessus (les numéros correspondent), et en ouvrant les mêmes coordonnées dans la BD ORTHO (QGIS, flux WMS de la Géoplateforme IGN).

| # | Sentier | Visible en BSI Sentinel-2 | Visible en BD ORTHO | Remarque |
| --- | --- | --- | --- | --- |
| 1 | [nom] | oui / non | oui / non | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |
| 6 | | | | |
| 7 | | | | |
| 8 | | | | |
| 9 | | | | |
| 10 | | | | |

**Résultat** : [X] sentiers sur 10 visibles en Sentinel-2, [Y] sur 10 en BD ORTHO.

### Interprétation : visibilité des sentiers

> **À rédiger par l'équipe.** Auteur·e : [Prénom]

Questions pour guider la rédaction (à supprimer ensuite) :
- Le profil de BSI monte-t-il près des sentiers ? L'écart est-il net ?
- Quelle source d'image retenir pour annoter les sentiers et entraîner un modèle de détection ?

## 6. Synthèse des résultats

In [ ]:
# Tableau récapitulatif de toutes les comparaisons sentier / témoin
synthese = resultats_sentiers[["zone", "indice", "periode", "sentier_moyenne", "temoin_moyenne",
                               "ecart_sentier_moins_temoin", "sentier_pixels", "temoin_pixels"]]
synthese.to_csv("resultats/synthese.csv", index=False)
synthese

### Conclusion

_À rédiger : la réponse à la question du carnet, en 5 à 10 lignes, chiffres à l'appui._

### Limites

_À rédiger. Pistes : résolution de 10 m face à des sentiers de 1 à 2 m ; sentiers absents d'OpenStreetMap ; routes et pistes non distinguées ;
zones perturbées (incendies, repousse) ; une seule saison étudiée ; pas de test statistique tenant compte de l'autocorrélation spatiale._

### Suite du travail

_À rédiger : prochaines analyses, données à obtenir, décisions à prendre avec le tuteur._

## 7. Contributions

| Membre | Partie du carnet | Contribution |
| --- | --- | --- |
| Axel Bonneau | [section] | [à compléter] |
| Sina Balaei Langroodi | [section] | [à compléter] |
| Jonas Barth | [section] | [à compléter] |
| Florian Berger | [section] | [à compléter] |
| Kuresa Ramen | [section] | [à compléter] |
| Enzo Sierra-Guisset | [section] | [à compléter] |

## Sources des données

- Images : Copernicus Sentinel-2 (ESA), via Google Earth Engine.
- Sentiers : © les contributeurs d'OpenStreetMap, données sous licence ODbL.